### Multiclass Classification Metrics

Multiclass classification predicts one label from **three or more possible classes**. Examples include classifying a flower species, recognizing a digit, or identifying a type of product.

This notebook explains:
- Confusion Matrix
- Accuracy
- Precision, Recall, and F1 Score for each class
- Macro, Micro, and Weighted averages
- Classification Report

We use Scikit-learn's **Iris dataset**, which contains three classes: Setosa, Versicolor, and Virginica.

### Load and Prepare the Dataset

The Iris dataset is built into Scikit-learn. It has four numeric input features and three target classes.

We use a stratified train/test split so that each class keeps approximately the same proportion in both sets. A scaled Logistic Regression pipeline is used as a simple baseline classifier.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, classification_report
)

iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

class_names = iris.target_names
print("Classes:", list(class_names))
print("Test accuracy:", accuracy_score(y_test, y_pred))

Classes: [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]
Test accuracy: 0.9333333333333333


### 1. Confusion Matrix

A multiclass confusion matrix has one row and one column for each class.

- **Rows** represent actual classes.
- **Columns** represent predicted classes.
- The diagonal cells contain correct predictions.
- Off-diagonal cells show which classes are being confused.

For example, a value in the Setosa row and Versicolor column means actual Setosa observations were predicted as Versicolor.

In [2]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])
cm_df = pd.DataFrame(
    cm,
    index=[f"Actual: {name}" for name in class_names],
    columns=[f"Predicted: {name}" for name in class_names]
)
cm_df

,Predicted: setosa,Predicted: versicolor,Predicted: virginica
Actual: setosa,10,0,0
Actual: versicolor,0,9,1
Actual: virginica,0,1,9


### 2. Accuracy

Accuracy is the proportion of all observations that were classified correctly.

$$
Accuracy = \\frac{\\text{Number of correct predictions}}{\\text{Total number of predictions}}
$$

For a multiclass confusion matrix, this is the sum of the diagonal values divided by the sum of all matrix values.

**When to use**
- When classes are reasonably balanced.
- When mistakes across classes have similar importance.

**Advantages**
- Easy to understand.
- Gives one overall measure of correct predictions.

**Disadvantages**
- Can hide poor performance on a smaller class.
- Does not show which classes are confused.

In [3]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.3f} ({accuracy:.1%})")
print("Manual calculation:", np.trace(cm) / np.sum(cm))

Accuracy: 0.933 (93.3%)
Manual calculation: 0.9333333333333333


### 3. Precision, Recall, and F1 for Each Class

In multiclass classification, precision, recall, and F1 are calculated **separately for each class** using a One-vs-Rest idea.

For a selected class:
- Treat that class as **positive**.
- Treat all other classes together as **negative**.
- Calculate TP, FP, FN, and TN for that class.
- Apply the binary formulas.

For class $i$:

$$
Precision_i = \\frac{TP_i}{TP_i + FP_i}
$$

$$
Recall_i = \\frac{TP_i}{TP_i + FN_i}
$$

$$
F1_i = 2 \\times \\frac{Precision_i \\times Recall_i}{Precision_i + Recall_i}
$$

**Interpretation**
- Precision for Setosa: among all predictions of Setosa, how many were actually Setosa?
- Recall for Setosa: among all actual Setosa samples, how many were correctly predicted?
- F1 for Setosa: harmonic mean of Setosa precision and recall.

The same process is repeated for Versicolor and Virginica.

In [4]:
for average_name, metric in [
    ("Precision", precision_score),
    ("Recall", recall_score),
    ("F1 Score", f1_score)
]:
    values = metric(y_test, y_pred, labels=[0, 1, 2], average=None, zero_division=0)
    print(average_name)
    display(pd.Series(values, index=class_names, name=average_name))

Precision


setosa        1.0
versicolor    0.9
virginica     0.9
Name: Precision, dtype: float64

Recall


setosa        1.0
versicolor    0.9
virginica     0.9
Name: Recall, dtype: float64

F1 Score


setosa        1.0
versicolor    0.9
virginica     0.9
Name: F1 Score, dtype: float64

### 4. Macro Average

Macro averaging calculates the metric separately for every class and then takes the simple average. Each class receives equal weight, regardless of how many samples it contains.

For precision:

$$
Macro\\ Precision = \\frac{Precision_1 + Precision_2 + \\cdots + Precision_K}{K}
$$

The same averaging process is used for recall and F1.

**When to use**
- When every class is equally important.
- Especially useful when class sizes are imbalanced and minority-class performance matters.

**Advantages**
- Gives every class equal influence.
- Makes poor performance on a small class more visible.

**Disadvantages**
- Does not account for class frequency.
- Can be strongly affected by a weak score on a very small class.

In [5]:
print("Macro Precision:", precision_score(y_test, y_pred, average="macro", zero_division=0))
print("Macro Recall:   ", recall_score(y_test, y_pred, average="macro", zero_division=0))
print("Macro F1:       ", f1_score(y_test, y_pred, average="macro", zero_division=0))

Macro Precision: 0.9333333333333332
Macro Recall:    0.9333333333333332
Macro F1:        0.9333333333333332


### 5. Micro Average

Micro averaging first adds the TP, FP, and FN counts across all classes, then calculates the metric from those global totals.

$$
Micro\\ Precision = \\frac{\\sum_i TP_i}{\\sum_i TP_i + \\sum_i FP_i}
$$

$$
Micro\\ Recall = \\frac{\\sum_i TP_i}{\\sum_i TP_i + \\sum_i FN_i}
$$

Micro F1 is calculated from micro precision and micro recall.

In standard single-label multiclass classification, each observation has exactly one true class and one predicted class. In this setting, micro precision, micro recall, micro F1, and accuracy are equal.

**When to use**
- When you want an overall metric that pools decisions across classes.
- Common in multiclass or multilabel settings where total decisions matter.

**Advantages**
- Uses all class-level TP, FP, and FN counts.
- Provides a single global summary.

**Disadvantages**
- In single-label multiclass classification, it equals accuracy and can hide weak minority-class performance.
- Large classes can contribute more decisions to the total.

In [6]:
print("Micro Precision:", precision_score(y_test, y_pred, average="micro", zero_division=0))
print("Micro Recall:   ", recall_score(y_test, y_pred, average="micro", zero_division=0))
print("Micro F1:       ", f1_score(y_test, y_pred, average="micro", zero_division=0))
print("Accuracy:       ", accuracy_score(y_test, y_pred))

Micro Precision: 0.9333333333333333
Micro Recall:    0.9333333333333333
Micro F1:        0.9333333333333333
Accuracy:        0.9333333333333333


### 6. Weighted Average

Weighted averaging calculates the metric for each class and then averages the class scores using each class's **support** (number of actual samples) as its weight.

For precision:

$$
Weighted\\ Precision =
\\frac{\\sum_i Support_i \\times Precision_i}{\\sum_i Support_i}
$$

Weighted recall and weighted F1 use the same support weights.

**When to use**
- When you want a summary that reflects the dataset's class distribution.
- When larger classes should contribute more to the overall score.

**Advantages**
- Accounts for class frequency.
- Gives a useful overall summary when class sizes differ.

**Disadvantages**
- Large classes dominate the result.
- A poor score on a minority class may be hidden.

In [7]:
print("Weighted Precision:", precision_score(y_test, y_pred, average="weighted", zero_division=0))
print("Weighted Recall:   ", recall_score(y_test, y_pred, average="weighted", zero_division=0))
print("Weighted F1:       ", f1_score(y_test, y_pred, average="weighted", zero_division=0))

Weighted Precision: 0.9333333333333333
Weighted Recall:    0.9333333333333333
Weighted F1:        0.9333333333333333


### Classification Report

Scikit-learn's classification report shows precision, recall, F1, and support for every class, followed by accuracy and macro and weighted averages.

- **Support :** Number of actual observations in each class.
- **Macro avg :** Every class has equal weight.
- **Weighted avg :** Each class is weighted by its support.
- **Micro avg :** In ordinary single-label multiclass classification, micro precision/recall/F1 equal accuracy, so Scikit-learn generally reports accuracy rather than a separate micro row.

Use the class-wise rows to find which classes are difficult for the model, not just the overall averages.

In [8]:
print(classification_report(
    y_test, y_pred,
    target_names=class_names,
    zero_division=0
))

              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.90      0.90      0.90        10
   virginica       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30



### Which Metric Should You Choose?

| Goal / situation | Metric to focus on | Reason |
|:--|:--|:--|
| Classes balanced, errors similarly costly | Accuracy | Overall proportion correct |
| Every class matters equally | Macro F1 | Gives each class equal influence |
| Class imbalance exists and frequency should count | Weighted F1 | Weights class scores by support |
| Need one global pooled measure | Micro F1 | Pools TP, FP, and FN across classes |
| Need to find a weak class | Per-class precision / recall / F1 | Shows performance class by class |
| Need to understand class confusion | Confusion Matrix | Shows actual-versus-predicted class counts |

There is no single best metric for every multiclass problem. Choose according to class balance and the cost of misclassifying each class.

### Key Takeaways

- A multiclass confusion matrix has one row and one column per class.
- Precision, recall, and F1 are first calculated for each class using a One-vs-Rest view.
- **Macro** gives every class equal weight.
- **Micro** pools counts across classes; in single-label multiclass classification it equals accuracy.
- **Weighted** averages class scores according to support.
- Always inspect class-wise metrics when some classes are more important or less frequent than others.